<a href="https://colab.research.google.com/github/kasturikirankumar1101-lab/AI_TOOLS/blob/main/LangTransWdiffModels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def build_system_prompt(source_language: str, target_language: str) -> str:
    """
    Returns a precise, language-aware system prompt for code conversion.
    Covers all combinations of: Java, C++, Python, Javascript, .NET (C#), COBOL.
    """

    # ── Per-language rules ─────────────────────────────────────────────────────
    language_rules = {

        "C++": """
LANGUAGE-SPECIFIC RULES — C++ (TARGET)
═══════════════════════════════════════
HEADERS: Include ALL required headers based on actual usage:
  std::vector          → #include <vector>
  std::string          → #include <string>
  std::map/unordered   → #include <map> / #include <unordered_map>
  std::thread          → #include <thread>
  std::mutex           → #include <mutex>
  std::chrono          → #include <chrono>
  std::accumulate      → #include <numeric>
  std::transform       → #include <algorithm>
  std::sort/find       → #include <algorithm>
  std::setprecision    → #include <iomanip>
  std::function        → #include <functional>
  std::optional        → #include <optional>
  std::filesystem      → #include <filesystem>
  std::cout/cin        → #include <iostream>
  std::fstream         → #include <fstream>
  std::sstream         → #include <sstream>
  std::exception       → #include <stdexcept>
  std::sqrt/pow        → #include <cmath>

STANDARD: Compile as C++17. Use std:: prefix on ALL standard library items.
NEVER use: `using namespace std;`

VOID FUNCTION TIMER PATTERN (MANDATORY — never assign void to auto):
  template <typename Func, typename... Args>
  auto timer(const std::string& name, Func&& func, Args&&... args) {
      auto start = std::chrono::high_resolution_clock::now();
      if constexpr (std::is_void_v<std::invoke_result_t<Func, Args...>>) {
          func(std::forward<Args>(args)...);
          auto end = std::chrono::high_resolution_clock::now();
          std::chrono::duration<double> elapsed = end - start;
          std::cout << "[TIMER] " << name << ": " << elapsed.count() << " sec\\n";
      } else {
          auto result = func(std::forward<Args>(args)...);
          auto end = std::chrono::high_resolution_clock::now();
          std::chrono::duration<double> elapsed = end - start;
          std::cout << "[TIMER] " << name << ": " << elapsed.count() << " sec\\n";
          return result;
      }
  }

MEMORY: Use RAII — prefer smart pointers (std::unique_ptr, std::shared_ptr) over raw new/delete.
THREADING: Protect shared data with std::mutex + std::lock_guard. Never access shared state without locks.
EXCEPTIONS: Use try/catch with std::exception. Never swallow exceptions silently.
TYPE MAPPING FROM OTHER LANGUAGES:
  int/Integer          → int or int64_t for large values
  float/double         → double
  bool/Boolean         → bool
  String               → std::string
  List/ArrayList       → std::vector<T>
  Map/HashMap          → std::unordered_map<K,V>
  null/None/null       → nullptr or std::optional<T>
  void method          → void function
  interface            → abstract class with pure virtual methods
  decorator            → wrapper function or template
""",

        "Python": """
LANGUAGE-SPECIFIC RULES — Python (TARGET)
══════════════════════════════════════════
VERSION: Python 3.10+. Use f-strings, type hints, dataclasses where appropriate.
IMPORTS: Only import what is used. Prefer stdlib before third-party.
  threading            → import threading
  file I/O             → built-in open() — no import needed
  math functions       → import math
  timing               → import time
  regex                → import re
  dataclasses          → from dataclasses import dataclass
  typing               → from typing import Optional, List, Dict, Tuple

STYLE: Follow PEP 8. snake_case for functions/variables. PascalCase for classes.
THREADING: Use threading.Thread. Protect shared state with threading.Lock().
EXCEPTIONS: Use try/except with specific exception types. Never bare `except:`.
MEMOIZATION: Use @functools.lru_cache or @functools.cache for recursive functions.
TYPE MAPPING FROM OTHER LANGUAGES:
  int/long/Integer     → int
  float/double/Double  → float
  bool/Boolean         → bool
  char/String          → str
  List/ArrayList/vector→ list
  Map/HashMap/dict     → dict
  Set/HashSet          → set
  null/nullptr/None    → None
  Optional<T>          → Optional[T] with type hint
  void                 → no return annotation or -> None
  interface/abstract   → ABC with @abstractmethod
  decorator pattern    → Python decorator with @functools.wraps
  try/catch            → try/except
  finally              → finally
""",

        "Java": """
LANGUAGE-SPECIFIC RULES — Java (TARGET)
════════════════════════════════════════
VERSION: Java 17+. Use modern features: records, var, switch expressions where appropriate.
IMPORTS: Import every class used. No wildcard imports (no `import java.util.*;`).
  List, ArrayList      → import java.util.ArrayList; import java.util.List;
  Map, HashMap         → import java.util.HashMap; import java.util.Map;
  Optional             → import java.util.Optional;
  Stream/Collectors    → import java.util.stream.*; import java.util.stream.Collectors;
  Thread               → import java.lang.Thread; (auto-imported)
  ExecutorService      → import java.util.concurrent.ExecutorService;
  AtomicInteger        → import java.util.concurrent.atomic.AtomicInteger;
  synchronized/Lock    → import java.util.concurrent.locks.ReentrantLock;
  File I/O             → import java.io.*; or java.nio.file.*
  Math functions       → import java.lang.Math; (auto-imported)

STYLE: PascalCase for classes. camelCase for methods/variables. ALL_CAPS for constants.
THREADING: Use ExecutorService/Thread. Synchronize shared state with synchronized or ReentrantLock.
EXCEPTIONS: Use try/catch/finally with specific exception types. Use checked vs unchecked correctly.
GENERICS: Always specify type parameters — never use raw types (e.g. List, not List<Object> unless needed).
TYPE MAPPING FROM OTHER LANGUAGES:
  int                  → int or Integer (boxed)
  float/double         → double or Double (boxed)
  bool/Boolean         → boolean or Boolean (boxed)
  string/String/str    → String
  list/vector          → List<T> / ArrayList<T>
  dict/map             → Map<K,V> / HashMap<K,V>
  None/null/nullptr    → null or Optional<T>
  pointer              → object reference
  struct               → class or record
  interface/abstract   → interface or abstract class
  decorator            → AOP or wrapper class pattern
  try/except/catch     → try/catch/finally
""",

        "Javascript": """
LANGUAGE-SPECIFIC RULES — JavaScript (TARGET)
══════════════════════════════════════════════
VERSION: ES2022+. Use modern syntax: const/let (never var), arrow functions, async/await.
MODULES: Use ES module syntax (import/export) for Node.js 14+, or CommonJS (require) if specified.
ASYNC: Convert threads/concurrent code to async/await with Promises. Use Promise.all for parallel work.
STYLE: camelCase for variables/functions. PascalCase for classes. UPPER_CASE for constants.
ERROR HANDLING: Use try/catch/finally. Never swallow errors silently.
FILE I/O (Node.js): Use fs.promises (async) — never synchronous fs.readFileSync in production.
  const fs = require('fs').promises;  // CommonJS
  import fs from 'fs/promises';       // ESM

THREADING NOTE: JavaScript is single-threaded. Convert threads to:
  - async/await + Promise for I/O concurrency
  - Worker Threads (worker_threads module) for CPU parallelism
TYPE MAPPING FROM OTHER LANGUAGES:
  int/long/double      → number
  bool/Boolean         → boolean
  String/string/str    → string (template literals for formatting)
  List/ArrayList/vector→ Array
  Map/HashMap/dict     → Map or plain object {}
  Set/HashSet          → Set
  null/None/nullptr    → null or undefined
  Optional<T>          → value ?? defaultValue pattern
  void                 → function with no return
  interface            → JSDoc typedef or TypeScript interface (if TS requested)
  class                → class (ES6+)
  try/except/catch     → try/catch/finally
""",

        ".NET": """
LANGUAGE-SPECIFIC RULES — C# / .NET (TARGET)
═════════════════════════════════════════════
VERSION: C# 10+ / .NET 6+. Use modern features: records, pattern matching, global usings, nullable refs.
NAMESPACES: Include ALL required using statements:
  Console.WriteLine    → using System;
  List<T>, Dictionary  → using System.Collections.Generic;
  Thread, Task         → using System.Threading; / using System.Threading.Tasks;
  File I/O             → using System.IO;
  Regex                → using System.Text.RegularExpressions;
  Stopwatch/DateTime   → using System.Diagnostics; / using System;
  Math functions       → using System; (Math is in System)
  LINQ                 → using System.Linq;

STYLE: PascalCase for classes/methods/properties. camelCase for local variables. _camelCase for private fields.
ASYNC: Use async/await with Task<T>. Convert threads to Task.Run() or async methods where possible.
THREADING: Use lock(obj){} for synchronization, or SemaphoreSlim/Mutex for async contexts.
EXCEPTIONS: Use try/catch/finally with specific exception types. Use when clause for filtering.
NULLABLE: Enable nullable reference types. Use T? for nullable types. Use ?? and ?.
TYPE MAPPING FROM OTHER LANGUAGES:
  int/Integer          → int (System.Int32)
  long/Long            → long (System.Int64)
  float                → float (System.Single)
  double/Double        → double (System.Double)
  bool/Boolean         → bool (System.Boolean)
  String/string/str    → string
  List/ArrayList/vector→ List<T>
  Map/HashMap/dict     → Dictionary<K,V>
  Set/HashSet          → HashSet<T>
  null/None/nullptr    → null or T? (nullable)
  Optional<T>          → T? or Nullable<T>
  interface            → interface IFoo { }
  abstract class       → abstract class
  decorator            → Attribute or wrapper method
  try/except/catch     → try/catch/finally
""",

        "COBOL": """
LANGUAGE-SPECIFIC RULES — COBOL (TARGET)
═════════════════════════════════════════
VERSION: COBOL 2002/2014 standard. Enterprise-grade, mainframe-compatible output.
STRUCTURE: ALL COBOL programs MUST include these four divisions in order:
  1. IDENTIFICATION DIVISION   — program name and metadata
  2. ENVIRONMENT DIVISION      — file and hardware configuration
  3. DATA DIVISION             — all variable declarations (WORKING-STORAGE, FILE)
  4. PROCEDURE DIVISION        — executable logic

NAMING: COBOL identifiers use KEBAB-CASE (e.g. CUSTOMER-NAME, TOTAL-AMOUNT).
         Max 30 characters per identifier. No special characters except hyphen.
LITERALS: Numeric literals are unquoted. String literals use single quotes: 'HELLO'.
ARITHMETIC: Use ADD, SUBTRACT, MULTIPLY, DIVIDE or COMPUTE for expressions.
  COMPUTE RESULT = (A + B) * C / D
STRINGS: Use STRING/UNSTRING for concatenation/splitting. MOVE for assignment.
LOOPS:
  PERFORM VARYING I FROM 1 BY 1 UNTIL I > LIMIT
  PERFORM paragraph-name TIMES / UNTIL condition
CONDITIONALS:
  IF condition
      statement
  ELSE
      statement
  END-IF
FILE I/O: Define files in ENVIRONMENT DIVISION (SELECT/ASSIGN). Open/Close explicitly.
  OPEN INPUT / OUTPUT / I-O file-name
  READ / WRITE / REWRITE / DELETE
  CLOSE file-name
ERROR HANDLING: Use file status codes (e.g. WS-FILE-STATUS) and ON EXCEPTION clauses.
PARAGRAPHS: Each logical section is a paragraph. STOP RUN terminates the program.
TYPE MAPPING FROM OTHER LANGUAGES:
  int/Integer          → PIC 9(n) COMP or PIC 9(n)
  long/large int       → PIC 9(18) COMP-3
  float/double         → PIC 9(n)V9(m) COMP-1 / COMP-2
  bool/Boolean         → PIC X(1) with values 'Y'/'N' or '1'/'0'
  String/str/string    → PIC X(n) — size must be declared explicitly
  List/Array           → OCCURS n TIMES table with INDEXED BY
  Map/HashMap          → OCCURS table with SEARCH/SEARCH ALL (binary search)
  null/None/nullptr    → SPACES (alphanumeric) or ZEROS (numeric)
  class/struct         → GROUP item in WORKING-STORAGE
  method/function      → SECTION or PARAGRAPH called via PERFORM
  exception/try-catch  → EVALUATE / ON EXCEPTION / INVALID KEY clauses
  print/output         → DISPLAY or WRITE to file
  thread               → Not natively supported; use batch sequencing or MQ
"""
    }

    # ── Language-specific source-to-target conversion hints ───────────────────
    conversion_hints = {
        ("Python", "C++"): """
PYTHON → C++ SPECIFIC MAPPINGS:
  @lru_cache / @cache      → std::unordered_map memo + lock_guard (non-reentrant safe)
  generator (yield)        → std::vector<T> with eager population, or custom iterator
  list comprehension       → std::transform / range-based for loop
  dict                     → std::unordered_map<K,V>
  tuple return             → std::pair<A,B> or std::tuple<A,B,C>
  *args/**kwargs           → variadic template or std::initializer_list
  with open() as f         → std::ifstream/ofstream with RAII (no explicit close needed)
  threading.Thread         → std::thread (always .join() before destruction)
  threading.Lock()         → std::mutex + std::lock_guard<std::mutex>
  try/except               → try/catch (std::exception& e)
  f-string                 → std::ostringstream or std::format (C++20)
  None                     → nullptr or std::optional<T>
  @decorator               → template wrapper function (use if constexpr for void safety)
""",
        ("Java", "C++"): """
JAVA → C++ SPECIFIC MAPPINGS:
  ArrayList<T>             → std::vector<T>
  HashMap<K,V>             → std::unordered_map<K,V>
  interface                → abstract class with pure virtual methods
  implements               → public inheritance
  synchronized             → std::mutex + std::lock_guard
  ExecutorService          → std::thread pool (manual) or std::async
  Optional<T>              → std::optional<T>
  try/catch/finally        → try/catch (no finally — use RAII destructors)
  null                     → nullptr or std::optional
  instanceof               → dynamic_cast<T*>
  String                   → std::string
  System.out.println       → std::cout
""",
        ("Javascript", "C++"): """
JAVASCRIPT → C++ SPECIFIC MAPPINGS:
  Array                    → std::vector<T>
  Object / Map             → std::unordered_map<std::string, T>
  Promise/async/await      → std::future / std::async (or std::thread for parallelism)
  typeof / instanceof      → typeid or static_cast checks
  undefined/null           → nullptr or std::optional
  console.log              → std::cout
  JSON.stringify           → manual serialisation or a library
  arrow functions          → lambdas: [captures](params){ body }
  spread operator ...      → std::initializer_list or variadic template
  try/catch                → try/catch (std::exception)
""",
        ("C++", "Python"): """
C++ → PYTHON SPECIFIC MAPPINGS:
  std::vector<T>           → list
  std::unordered_map<K,V>  → dict
  std::optional<T>         → Optional[T] with None check
  std::thread              → threading.Thread
  std::mutex + lock_guard  → threading.Lock() with `with lock:`
  std::cout                → print()
  std::ifstream/ofstream   → open() context manager
  template<typename T>     → Generic type hint or duck typing
  nullptr                  → None
  try/catch                → try/except
  pure virtual method      → @abstractmethod in ABC subclass
  destructor (~Class)      → __del__ or context manager (__enter__/__exit__)
""",
        ("COBOL", "Python"): """
COBOL → PYTHON SPECIFIC MAPPINGS:
  WORKING-STORAGE items    → module-level or class variables
  PIC 9(n)                 → int
  PIC X(n)                 → str (strip trailing spaces)
  OCCURS n TIMES           → list of fixed size or dict
  PERFORM paragraph        → function call
  PERFORM VARYING          → for loop
  EVALUATE / WHEN          → if/elif/else or match/case (Python 3.10+)
  MOVE value TO var        → var = value
  COMPUTE expr             → standard arithmetic expression
  DISPLAY                  → print()
  OPEN/READ/WRITE/CLOSE    → open() context manager
  STOP RUN                 → sys.exit(0) or end of main()
  FILE STATUS codes        → exception handling with try/except
""",
    }

    # Fetch language-specific target rules
    target_rules = language_rules.get(target_language, "")

    # Fetch conversion-specific hints (if pair exists)
    hint_key = (source_language, target_language)
    conversion_hint = conversion_hints.get(hint_key, "")

    system_prompt = f"""
You are an expert compiler-engineer and {target_language} specialist with deep knowledge of {source_language}.
Your ONLY job is to convert the given {source_language} code to flawless, production-ready {target_language} code.

═══════════════════════════════════════════════════════════════
ABSOLUTE REQUIREMENTS — VIOLATIONS WILL CAUSE REJECTION
═══════════════════════════════════════════════════════════════

1. CORRECTNESS
   ✦ The converted code MUST be 100% functionally equivalent to the source.
   ✦ Every function, class, method, and logic branch must be present.
   ✦ Do NOT omit, stub, simplify, or summarise any part of the code.
   ✦ Do NOT write comments like "// rest of implementation" or "# TODO".

2. COMPILABILITY / RUNNABILITY
   ✦ The output MUST compile (compiled languages) or run (interpreted languages) without errors.
   ✦ Verify every identifier is declared before use.
   ✦ Verify every import/include is present before referencing it.
   ✦ If a direct equivalent does not exist, implement the closest correct alternative.

3. IMPORTS / HEADERS / NAMESPACES
   ✦ Include EVERY required import, #include, using, or require statement.
   ✦ Do NOT include unused imports.
   ✦ Follow the exact import style of the target language.

4. LANGUAGE IDIOMS
   ✦ Write idiomatic {target_language} — not a literal line-by-line transliteration.
   ✦ Use the standard library of {target_language} wherever applicable.
   ✦ Follow naming conventions, formatting, and patterns native to {target_language}.

5. ERROR HANDLING
   ✦ Preserve ALL exception/error handling from the source code.
   ✦ Use the correct error-handling idiom of {target_language}.
   ✦ Never silently swallow exceptions.

6. CONCURRENCY / THREADING
   ✦ Convert threading/concurrency constructs to the correct {target_language} equivalent.
   ✦ Protect all shared state with the appropriate synchronisation primitive.

7. TRANSPARENCY
   ✦ Add a brief inline comment ONLY where the translation is non-obvious.
   ✦ Mark genuine untranslateable patterns: // ASSUMPTION: <reason>
   ✦ Mark real limitations: // LIMITATION: <reason>

{target_rules}
{conversion_hint}

═══════════════════════════════════════════════════════════════
MANDATORY PRE-OUTPUT VALIDATION CHECKLIST
═══════════════════════════════════════════════════════════════

Before writing the final output, mentally verify ALL of the following:

  ☐ Every function from the source is present in the output
  ☐ Every import/header needed is included
  ☐ No undefined variables or symbols
  ☐ No syntax errors (missing brackets, semicolons, indentation)
  ☐ Threading/concurrency is correctly synchronised
  ☐ Error handling is preserved
  ☐ No stub code, no TODOs, no placeholder comments
  ☐ Output compiles/runs with zero errors

If ANY item fails → FIX IT before producing output.

═══════════════════════════════════════════════════════════════
OUTPUT FORMAT — STRICT, NO EXCEPTIONS
═══════════════════════════════════════════════════════════════

<converted_code language="{target_language}">
ONLY raw {target_language} code
NO markdown code fences (no ``` or ```{target_language.lower()})
NO explanations before or after the tag
NO preamble, NO summary, NO closing remarks
</converted_code>
"""
    return system_prompt.strip()


# ── User prompt (unchanged structure) ─────────────────────────────────────────
def build_user_prompt(source_language: str, target_language: str, code_snippet: str) -> str:
    return f"""Convert the following {source_language} code to {target_language}.

STRICT REQUIREMENTS:
- Output must be 100% complete — no omissions.
- Output must compile/run without errors.
- Preserve all logic, structure, and behaviour exactly.
- Do NOT generate pseudo-code or placeholders.

Input Code ({source_language}):
{code_snippet}"""


# ── Full prompt builder (drop-in replacement for existing build_prompt()) ─────
def build_prompts(source_language: str, target_language: str, code_snippet: str) -> tuple[str, str]:
    """
    Assembles system and user prompts.
    Returns them as a tuple (system, user) to ensure clean API separation.
    """
    # Assuming build_system_prompt and build_user_prompt are defined as provided
    system_prompt = build_system_prompt(source_language, target_language)
    user_prompt = build_user_prompt(source_language, target_language, code_snippet)

    return system_prompt, user_prompt




In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
def quantization_model(model_name):
  # model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# Load tokenizer
  tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model in 4-bit
  model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True,          # 🔥 quantization happens here
    bnb_4bit_compute_dtype=torch.float16
)

  return model



In [3]:
def model_gpt_oss_120b(system_prompt:str, user_prompt:str,  model_name:str):

  from transformers import pipeline
  import torch


  model = quantization_model(model_name)
  pipe = load_model(model)

  messages = [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": user_prompt},
  ]

  outputs = pipe(
      messages,
      max_new_tokens=256,
  )
  return outputs[0]["generated_text"][-1]

In [9]:
"""
Code Converter — Production-Grade Gradio App
Converts source code between languages using OpenAI GPT API.
Runs in Google Colab (reads key from Colab Secrets) or locally (reads from env var).
"""
#--------------------------------------------------------------------
# This program uses the GPT closed source model by caling API
# ___________________________________________________________________

import os
import re
import logging
import gradio as gr
from openai import OpenAI, APIError, APIConnectionError, RateLimitError, AuthenticationError
from google.colab import userdata
from huggingface_hub import login
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks')
import model_manager

#__ HF_KEY read to login into Hugging face______________________________________

# -----------------------------
# 🔐 Login to Hugging Face
# -----------------------------
hf_token = userdata.get('HF_TOKEN')

if hf_token and hf_token.startswith("hf_"):
    print("✅ HF key looks good")
    login(hf_token)
else:
    print("❌ HF key missing")



# ── Colab Secrets (preferred) with os.environ fallback ────────────────────────
try:
    from google.colab import userdata as colab_userdata
    _COLAB = True
except ImportError:
    _COLAB = False

# ── Logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

if _COLAB:
    logger.info("Running inside Google Colab — API key will be read from Colab Secrets.")
else:
    logger.info("Running outside Google Colab — API key will be read from environment variable.")

# ── Constants ──────────────────────────────────────────────────────────────────
LANGUAGES      = ["Java", "Python", "Javascript", "C++", "COBOL", "VC++", ".NET"]
MODELS_LIST = ["gpt-oss-120B(O)", "deepseek-coder-v2(O)", "qwen/qwen3-coder-30b-a3b-instruct(O)",
               "claude-sonnet-4-5-20250929(C)", "GPT-5(C)", "GPT-oss-20B(O)", "gemini-2.5-pro(C)", "gpt-4o" ]
MODEL          = "gpt-4o"    # change to "gpt-4-turbo" or "gpt-3.5-turbo" if needed
MAX_TOKENS     = 8192        # enough for large file conversions
MAX_INPUT_CHARS = 100_000    # ~100 KB safety cap on input


# ── API Client ─────────────────────────────────────────────────────────────────
def get_client() -> OpenAI:
    """
    Resolve the OpenAI API key and return an OpenAI client.

    Resolution order:
      1. Google Colab Secrets  (key name: OPENAI_API_KEY)  — when running in Colab
      2. Environment variable   OPENAI_API_KEY             — fallback / local runs

    To add the key in Colab:
      - Click the 🔑 key icon in the left sidebar
      - Add a secret named  OPENAI_API_KEY  with your key value
      - Toggle "Notebook access" ON
    """
    api_key = ""

    # 1️⃣  Try Colab Secrets first
    if _COLAB:
        try:
            api_key = colab_userdata.get("OPENAI_API_KEY").strip()
            logger.info("✅ API key loaded from Colab Secrets.")
        except Exception as e:
            logger.warning("Colab Secrets lookup failed (%s). Falling back to env var.", e)

    # 2️⃣  Fall back to environment variable
    if not api_key:
        api_key = os.environ.get("OPENAI_API_KEY", "").strip()
        if api_key:
            logger.info("✅ API key loaded from environment variable.")

    # 3️⃣  Neither source had a key — raise a clear error
    if not api_key:
        raise EnvironmentError(
            "OPENAI_API_KEY not found.\n"
            "• In Colab : open the 🔑 Secrets panel and add OPENAI_API_KEY.\n"
            "• Locally  : run  export OPENAI_API_KEY=sk-...  before launching."
        )

    if not api_key.startswith("sk-"):
        logger.warning("OPENAI_API_KEY does not start with 'sk-' — double-check the value.")

    return OpenAI(api_key=api_key)





# ── Model Call ─────────────────────────────────────────────────────────────────
def call_model(system_prompt: str, user_prompt: str, selected_model: str) -> str:
    """
    Call the OpenAI API and return the assistant's text response.
    Raises descriptive RuntimeError on failure.
    """
    if selected_model == "gpt-4o":
      response = model_gpt5(system_prompt, user_prompt, selected_model)
      return response
    elif selected_model == "gpt-oss-120B(O)":
      selected_model = "openai/gpt-oss-120b"
      response = model_gpt_oss_120b(system_prompt, user_prompt, selected_model)
      return response




# ── Response Parser ────────────────────────────────────────────────────────────
def extract_converted_code(raw_response: str) -> str:
    """
    Pull the code out of the <converted_code ...>...</converted_code> tag.
    Falls back to returning the full response if the tag is absent.
    """
    match = re.search(
        r"<converted_code[^>]*>(.*?)</converted_code>",
        raw_response,
        re.DOTALL,
    )
    if match:
        return match.group(1).strip()
    logger.warning("Could not find <converted_code> tag in model response; returning raw output.")
    return raw_response.strip()
# __C++ Code Execution_________________________________________________________
def execute_cplus_code(cpcode: str) -> str:
  import gradio as gr
  import subprocess
  import tempfile
  import os
  import time   # ✅ NEW
  try:
        # Create temporary file
        with tempfile.NamedTemporaryFile(delete=False, suffix=".cpp") as f:
            cpp_file = f.name
            f.write(cpcode.encode())

        exe_file = cpp_file.replace(".cpp", "")

        # Compile
        compile_process = subprocess.run(
            ["g++", cpp_file, "-o", exe_file],
            capture_output=True,
            text=True
        )

        if compile_process.returncode != 0:
            return f"❌ Compilation Error:\n{compile_process.stderr}"

        # ✅ Start timer
        start_time = time.time()

        # Run executable
        run_process = subprocess.run(
            [exe_file],
            capture_output=True,
            text=True
        )

        # ✅ End timer
        end_time = time.time()

        execution_time = end_time - start_time

        return f"""✅ Output:
{run_process.stdout}

⏱️ Execution Time: {execution_time:.6f} seconds
"""

  except Exception as e:
        return f"⚠️ Error: {str(e)}"

  finally:
        # Cleanup
        try:
            os.remove(cpp_file)
            if os.path.exists(exe_file):
                os.remove(exe_file)
        except:
            pass

# __Java Code Execution_________________________________________________________

import subprocess
import tempfile
import os
import time
import re

def execute_java_code(javacode):
    try:
        # ✅ Extract class name (very important in Java)
        match = re.search(r'public\s+class\s+(\w+)', javacode)
        if not match:
            return "❌ Error: Please define a public class (e.g., public class Main)"

        class_name = match.group(1)

        # ✅ Create temp directory
        temp_dir = tempfile.mkdtemp()
        java_file = os.path.join(temp_dir, f"{class_name}.java")

        # Write code to file
        with open(java_file, "w") as f:
            f.write(javacode)

        # ✅ Compile
        compile_process = subprocess.run(
            ["javac", java_file],
            capture_output=True,
            text=True
        )

        if compile_process.returncode != 0:
            return f"❌ Compilation Error:\n{compile_process.stderr}"

        # ✅ Start timer
        start_time = time.perf_counter()

        # ✅ Run
        run_process = subprocess.run(
            ["java", "-cp", temp_dir, class_name],
            capture_output=True,
            text=True
        )

        # ✅ End timer
        end_time = time.perf_counter()

        execution_time = end_time - start_time

        return f"""✅ Output:
{run_process.stdout}

⏱️ Execution Time: {execution_time:.6f} seconds
"""

    except Exception as e:
        return f"⚠️ Error: {str(e)}"

    finally:
        # Cleanup
        try:
            for file in os.listdir(temp_dir):
                os.remove(os.path.join(temp_dir, file))
            os.rmdir(temp_dir)
        except:
            pass

# ________________________________________________________________________________

#__Clean code before executing____________________________________________________
def clean_code_block(code: str) -> str:
    if not code or not code.strip():
        return code

    lines = code.strip().splitlines()

    # ----------------------------
    # Case 1: Remove Markdown ``` blocks
    # ----------------------------
    if len(lines) >= 2:
        first = lines[0].strip()
        last = lines[-1].strip()

        if first.startswith("```") and last == "```":
            lines = lines[1:-1]

    # ----------------------------
    # Case 2: Remove triple quotes ''' or """
    # ----------------------------
    if len(lines) >= 2:
        first = lines[0].strip()
        last = lines[-1].strip()

        if (first in ["'''", '"""']) and (last == first):
            lines = lines[1:-1]

    return "\n".join(lines).strip()

#_______________________________________________________________________________

# __Python Code Execution_________________________________________________________
def execute_python_code(pycode: str) -> str:
    import io
    import contextlib
    import traceback
    import time
    import ast

    if not pycode.strip():
        return "⚠️ No code provided."

    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()
    exec_globals = {}

    start_time = time.perf_counter()

    try:
        # ── Step 1: Parse the code ──
        tree = ast.parse(pycode)

        # ── Step 2: Handle the last expression (Notebook-style) ──
        # If the last statement is an expression (like just 'x'),
        # we convert it to 'print(x)' so it shows up in stdout.
        if tree.body and isinstance(tree.body[-1], ast.Expr):
            last_expr = tree.body[-1]
            # Wrap the expression in a print() call
            print_node = ast.Expr(
                value=ast.Call(
                    func=ast.Name(id='print', ctx=ast.Load()),
                    args=[last_expr.value],
                    keywords=[]
                )
            )
            tree.body[-1] = print_node
            ast.fix_missing_locations(tree)

        # ── Step 3: Execute ──
        compiled_code = compile(tree, filename="<string>", mode="exec")

        with contextlib.redirect_stdout(stdout_buffer), \
             contextlib.redirect_stderr(stderr_buffer):
            exec(compiled_code, exec_globals)

        end_time = time.perf_counter()

        stdout_output = stdout_buffer.getvalue().strip()
        stderr_output = stderr_buffer.getvalue().strip()
        execution_time = end_time - start_time

        # ✅ Build clean response
        response = "📤 Output:\n"
        # If the user didn't print anything AND our auto-print didn't trigger, show 'No output'
        response += f"{stdout_output if stdout_output else 'No output (Note: use print() for intermediate values)'}\n"

        if stderr_output:
            response += f"\n⚠️ Warnings/Errors (stderr):\n{stderr_output}\n"

        response += f"\n⏱ Execution Time: {execution_time:.6f} seconds"
        return response

    except Exception:
        end_time = time.perf_counter()
        execution_time = end_time - start_time
        return (
            f"❌ Exception Occurred:\n{traceback.format_exc()}\n"
            f"⏱ Execution Time: {execution_time:.6f} seconds"
        )

# __Code to execute Javascript code______________________________________________


import subprocess
import tempfile
import time
import os

def execute_javascript_code(code: str) -> str:
    """
    Executes JavaScript code using Node.js and returns:
    - Output
    - Execution time
    - Errors (if any)
    """

    if not code.strip():
        return "⚠️ No code provided."

    try:
        # Create temporary JS file
        with tempfile.NamedTemporaryFile(delete=False, suffix=".js", mode="w") as f:
            js_file = f.name
            f.write(code)

        start_time = time.perf_counter()

        # Run JS using Node.js
        result = subprocess.run(
            ["node", js_file],
            capture_output=True,
            text=True,
            timeout=10   # ⏱ prevent infinite loops
        )

        end_time = time.perf_counter()
        execution_time = end_time - start_time

        output = result.stdout.strip()
        error = result.stderr.strip()

        # Cleanup temp file
        os.remove(js_file)

        if result.returncode != 0:
            return (
                f"❌ Error:\n{error if error else 'Unknown error'}\n\n"
                f"⏱ Execution Time: {execution_time:.6f} seconds"
            )

        return (
            f"📤 Output:\n"
            f"{output if output else 'No output'}\n\n"
            f"⏱ Execution Time: {execution_time:.6f} seconds"
        )

    except subprocess.TimeoutExpired:
        return "⏳ Execution timed out (possible infinite loop)."

    except Exception as e:
        return f"❌ Unexpected error:\n{str(e)}"




# __Code Execution logic_________________________________________________________
def code_execute(input_lang_code, lang_type):

  input_lang_code = clean_code_block(input_lang_code)
  print(f"clean code = {input_lang_code}")
  if lang_type == 'Python':
    exec_results = execute_python_code(input_lang_code)

  elif lang_type == 'C++':
    exec_results = execute_cplus_code(input_lang_code)

  elif lang_type == 'Java':
    exec_results = execute_java_code(input_lang_code)

  elif lang_type == 'Javascript':
    exec_results = execute_javascript_code(input_lang_code)



  return exec_results


# ── Dynamic Textbox Helper ─────────────────────────────────────────────────────
def dynamic_update(text: str, min_lines: int = 5, max_lines: int = 60) -> gr.update:
    """Return a gr.update that sets value and auto-sizes the Textbox."""
    lines = text.splitlines()
    display_lines = sum(max(1, (len(l) // 100) + 1) for l in lines)
    clamped = max(min_lines, min(display_lines, max_lines))
    return gr.update(value=text, lines=clamped)


# ── Main Processing Function ───────────────────────────────────────────────────
def process_code(
    code_input: str,
    convert_from: str,
    convert_to: str,selected_model,
) -> gr.update:
    """

    Gradio handler. Accepts raw code from the input Textbox,
    converts it using the GPT model, and returns a dynamic Textbox update.
    """
    print(f"selected Model = {selected_model}")
    # ── Input validation ───────────────────────────────────────────────────────
    if not code_input or not code_input.strip():
        return dynamic_update("⚠️  Input is empty. Please paste your source code and try again.")
    if not convert_from:
        return dynamic_update("⚠️  Please select a 'Convert From' language.")
    if not convert_to:
        return dynamic_update("⚠️  Please select a 'Convert To' language.")
    if convert_from == convert_to:
        return dynamic_update(
            "⚠️  'Convert From' and 'Convert To' are the same language. "
            "Please select different languages."
        )
    if len(code_input) > MAX_INPUT_CHARS:
        return dynamic_update(
            f"⚠️  Input is too large ({len(code_input):,} chars). "
            f"Maximum allowed is {MAX_INPUT_CHARS:,} chars."
        )

    # ── Build prompts & call model ─────────────────────────────────────────────
    try:
        system_prompt, user_prompt = build_prompts(convert_from, convert_to, code_input.strip())
        logger.info("Calling model: %s → %s (%d chars)", convert_from, convert_to, len(code_input))
        raw_response   = call_model(system_prompt, user_prompt, selected_model)
        converted_code = extract_converted_code(raw_response)
        logger.info("Conversion complete (%d chars).", len(converted_code))
    except RuntimeError as e:
        return dynamic_update(f"❌ {e}")
    except Exception as e:
        logger.exception("Unexpected error during conversion.")
        return dynamic_update(f"❌ Unexpected error:\n{e}")

    return dynamic_update(converted_code)


# ── UI Layout ──────────────────────────────────────────────────────────────────
with gr.Blocks(title="Code Converter", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # 🔄 Code Converter
        Paste your source code, choose the languages, and click **Convert**.
        The output area resizes automatically to fit the result.
        """
    )

    with gr.Row():
        # ── Left panel: dropdowns + input code ────────────────────────────────
        with gr.Column(scale=1):
            convert_from = gr.Dropdown(
                label="Convert From",
                choices=LANGUAGES,
                value=None,
                interactive=True,
            )
            convert_to = gr.Dropdown(
                label="Convert To",
                choices=LANGUAGES,
                value=None,
                interactive=True,
            )
            code_input = gr.Textbox(
                label="Source Code",
                lines=20,
                max_lines=60,
                interactive=True,
                placeholder="Paste your source code here…",
                show_copy_button=True,
            )


        # ── Right panel: converted output ──────────────────────────────────────
        with gr.Column(scale=1):
            output_box = gr.Textbox(
                label="Converted Code",
                lines=20,
                max_lines=60,
                interactive=False,
                placeholder="Converted code will appear here…",
                show_copy_button=True,
            )

    with gr.Row():

      input_code_execute = gr.Button("Execute Input Code", variant="primary", size="lg")

      model_selected = gr.Dropdown(
          label="Models List: ",
          choices=MODELS_LIST,
          value=MODELS_LIST[-1],
          interactive=True,
          elem_id="my_dropdown",
          scale=2
      )




      convert_btn = gr.Button("Convert", variant="primary", size="lg")
      output_code_execute = gr.Button("Execute output", variant="primary", size="lg")

    with gr.Row():
      input_exec_results = gr.Textbox(
          label="Input Code Execution results",
          lines=10,
          placeholder="Input code execution...",
          show_copy_button=True,
          scale=1
      )

      spacer = gr.Markdown("")  # empty spacer
      spacer.scale = 3          # 👈 creates space in the middle

      output_exec_results = gr.Textbox(
          label="Converted Code Execution results",
          lines=10,
          placeholder="converted code exection....",
          show_copy_button=True,
          scale=1
      )

    # ── Event wiring ───────────────────────────────────────────────────────────
    convert_btn.click(
        fn=process_code,
        inputs=[code_input, convert_from, convert_to, model_selected],
        outputs=output_box,
    )

    input_code_execute.click(
        fn=code_execute,
        inputs=[code_input, convert_from],
        outputs=input_exec_results,
    )

    output_code_execute.click(
        fn=code_execute,
        inputs=[output_box, convert_to],
        outputs=output_exec_results,
    )

    gr.Markdown(
        """<sub>
        🔑 <b>Colab users</b>: add <code>OPENAI_API_KEY</code> in the Secrets panel (left sidebar) and enable Notebook access.<br>
        💻 <b>Local users</b>: run <code>export OPENAI_API_KEY=sk-...</code> before launching.<br>
        🤖 Model in use: <code>gpt-4o</code> — change the <code>MODEL</code> constant at the top of the file to switch models.
        </sub>"""
    )


# In Colab, share=True creates a public tunnel URL.
# server_port is omitted so Gradio auto-selects a free port.
demo.launch(
    share=True, debug=True  # set False if running locally; share=True gives a public tunnel URL in Colab
)

✅ HF key looks good


/tmp/ipykernel_14944/2592999481.py:514: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Code Converter", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b04c4a966883c70ff4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


selected Model = gpt-oss-120B(O)


config.json: 0.00B [00:00, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

MXFP4 quantization requires Triton and kernels installed: CUDA requires Triton >= 3.4.0, XPU requires Triton >= 3.5.0, we will default to dequantizing the model to bf16


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b04c4a966883c70ff4.gradio.live


In [4]:
def model_gpt5(system_prompt: str, user_prompt: str, selected_model: str):

  client = get_client()

  try:
      response = client.chat.completions.create(
          model=selected_model,
          max_tokens=MAX_TOKENS,
          messages=[
              {"role": "system", "content": system_prompt},
              {"role": "user",   "content": user_prompt},
          ],
      )
  except AuthenticationError:
      raise RuntimeError(
          "❌ Invalid OpenAI API key.\n"
          "Please check your key at https://platform.openai.com/api-keys"
      )
  except RateLimitError:
      raise RuntimeError(
          "⏳ Rate limit or quota reached.\n"
          "Please check your usage at https://platform.openai.com/usage"
      )
  except APIConnectionError:
      raise RuntimeError("Could not connect to the OpenAI API. Check your network.")
  except APIError as e:
      if "insufficient_quota" in str(e) or "billing" in str(e).lower():
          raise RuntimeError(
              "💳 Your OpenAI credit balance is too low.\n"
              "Please visit https://platform.openai.com/settings/billing to add credits."
          )
      raise RuntimeError(f"OpenAI API error: {e}")

  # Extract text from the response
  return response.choices[0].message.content or ""